# Data Lake — Zonas, Formatos y Particionamiento en Python

## Unidad 4: Infraestructura de Datos

Este notebook construye un Data Lake local completo: ingestamos datos crudos de multiples fuentes y formatos (CSV, JSON, logs), los procesamos por las zonas Bronze → Silver → Gold, y comparamos formatos de almacenamiento (CSV vs Parquet). Tambien implementamos particionamiento y un catalogo de datos basico.

### Contenido:
1. Crear la estructura del Lake (zonas)
2. Bronze: ingestar datos crudos de multiples fuentes
3. Silver: limpiar y unificar
4. Gold: agregar para consumo
5. Particionamiento
6. Formatos: CSV vs Parquet — benchmark
7. Almacenamiento columnar: lectura selectiva
8. Catalogo de datos

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import shutil
import time
from datetime import datetime

---
## 1. Crear la estructura del Lake

In [ ]:
# ============================================================
# CREAR ESTRUCTURA DE CARPETAS
# ============================================================

# Un Data Lake es simplemente carpetas organizadas
# En produccion estas carpetas estan en S3 o Azure Blob
# Aqui usamos el filesystem local — los conceptos son iguales

LAKE_ROOT = 'datalake'

# Limpiar si existe
if os.path.exists(LAKE_ROOT):
    shutil.rmtree(LAKE_ROOT)

zonas = {
    'bronze': 'Datos crudos, tal como llegaron. Fuente de verdad.',
    'silver': 'Datos limpios, tipados, deduplicados. Listos para analisis.',
    'gold':   'Datos agregados y modelados. Listos para consumo (dashboards, KPIs).',
}

for zona, descripcion in zonas.items():
    os.makedirs(f'{LAKE_ROOT}/{zona}', exist_ok=True)
    # Dejar un README en cada zona
    with open(f'{LAKE_ROOT}/{zona}/README.md', 'w') as f:
        f.write(f'# Zona {zona.upper()}\n\n{descripcion}\n')

print("Estructura del Data Lake:")
print(f"  {LAKE_ROOT}/")
for zona in zonas:
    print(f"    {zona}/")
    print(f"      README.md")

---
## 2. Bronze: ingestar datos crudos

En Bronze guardamos los datos exactamente como llegan. No se limpian, no se transforman, no se borran. Cada fuente tiene su propia carpeta y los archivos se nombran con la fecha de llegada.

In [ ]:
# ============================================================
# FUENTE 1: CSV del sistema ERP (ventas estructuradas)
# ============================================================

os.makedirs(f'{LAKE_ROOT}/bronze/ventas_erp', exist_ok=True)

np.random.seed(42)

# Simular 3 dias de datos del ERP
for dia in ['2024-06-01', '2024-06-02', '2024-06-03']:
    n = np.random.randint(50, 100)
    df = pd.DataFrame({
        'fecha': dia,
        'producto': np.random.choice(['Dashboard', 'Reporte', 'API REST', 'App Web'], n),
        'region': np.random.choice(['Bogota', 'Medellin', 'Cali', 'Manizales'], n, p=[0.4, 0.25, 0.2, 0.15]),
        'unidades': np.random.randint(1, 30, n),
        'precio': np.round(np.random.uniform(200, 800, n), 2),
    })
    ruta = f'{LAKE_ROOT}/bronze/ventas_erp/{dia}.csv'
    df.to_csv(ruta, index=False)
    print(f"  Ingestado: {ruta} ({n} filas)")

In [ ]:
# ============================================================
# FUENTE 2: JSON de la API del CRM (clientes y leads)
# ============================================================

# Diferente fuente, diferente formato, diferentes columnas
# Esto es lo normal en un Data Lake: cada fuente manda lo que quiere

os.makedirs(f'{LAKE_ROOT}/bronze/crm_api', exist_ok=True)

crm_data = [
    {
        "client_name": "TechCorp",
        "contact_email": "ana@techcorp.co",
        "deal_value": 15000,
        "stage": "closed_won",
        "product_interest": "dashboard",
        "city": "Bogota",
        "created_at": "2024-06-01T10:23:45Z"
    },
    {
        "client_name": "DataSoft",
        "contact_email": "carlos@datasoft.com",
        "deal_value": 8500,
        "stage": "negotiation",
        "product_interest": "api",
        "city": "Medellin",
        "created_at": "2024-06-01T14:05:12Z"
    },
    {
        "client_name": "Analitika",
        "contact_email": "maria@analitika.co",
        "deal_value": 22000,
        "stage": "closed_won",
        "product_interest": "pipeline etl",
        "city": "Cali",
        "created_at": "2024-06-02T09:15:30Z"
    },
]

ruta_crm = f'{LAKE_ROOT}/bronze/crm_api/deals_2024-06.json'
with open(ruta_crm, 'w') as f:
    json.dump(crm_data, f, indent=2)

print(f"  Ingestado: {ruta_crm} ({len(crm_data)} registros)")

In [ ]:
# ============================================================
# FUENTE 3: Logs del servidor (texto semi-estructurado)
# ============================================================

os.makedirs(f'{LAKE_ROOT}/bronze/server_logs', exist_ok=True)

logs = [
    '2024-06-01 08:15:23 INFO  user=ana action=login ip=192.168.1.10',
    '2024-06-01 08:16:01 ERROR user=carlos action=api_call msg="timeout after 30s"',
    '2024-06-01 08:17:45 INFO  user=ana action=generate_report report_id=RPT-2024-001',
    '2024-06-01 08:20:12 WARN  user=maria action=upload msg="file too large (52MB)"',
    '2024-06-01 08:22:33 INFO  user=carlos action=login ip=192.168.1.15',
    '2024-06-01 08:25:00 INFO  user=ana action=export format=parquet rows=15000',
    '2024-06-01 08:30:18 ERROR user=pedro action=dashboard_load msg="query timeout"',
    '2024-06-01 08:35:44 INFO  user=ana action=logout session_duration=1221s',
]

ruta_logs = f'{LAKE_ROOT}/bronze/server_logs/app_2024-06-01.log'
with open(ruta_logs, 'w') as f:
    f.write('\n'.join(logs))

print(f"  Ingestado: {ruta_logs} ({len(logs)} lineas)")

In [ ]:
# ============================================================
# FUENTE 4: Encuesta de satisfaccion (Excel)
# ============================================================

os.makedirs(f'{LAKE_ROOT}/bronze/encuestas', exist_ok=True)

encuestas = pd.DataFrame({
    'Fecha Respuesta': ['01/06/2024', '02/06/2024', '02/06/2024', '03/06/2024'],
    'Cliente': ['TechCorp', 'DataSoft', 'Analitika', 'TechCorp'],
    'Producto Evaluado': ['Dashboard Ejecutivo', 'API REST', 'Pipeline ETL', 'Dashboard Ejecutivo'],
    'NPS (0-10)': [9, 6, 8, 10],
    'Comentario': [
        'Excelente, lo usamos todos los dias',
        'Funciona pero es lento a veces',
        'Muy util para automatizar',
        'Mejor que la version anterior'
    ],
})

ruta_encuesta = f'{LAKE_ROOT}/bronze/encuestas/nps_junio_2024.xlsx'
encuestas.to_excel(ruta_encuesta, index=False)

print(f"  Ingestado: {ruta_encuesta} ({len(encuestas)} respuestas)")

In [ ]:
# ============================================================
# VISTA GENERAL DEL BRONZE
# ============================================================

print("Contenido de Bronze:\n")
total_bytes = 0
for root, dirs, files in os.walk(f'{LAKE_ROOT}/bronze'):
    level = root.replace(f'{LAKE_ROOT}/bronze', '').count(os.sep)
    indent = '  ' * (level + 1)
    dirname = os.path.basename(root)
    if dirname != 'bronze':
        print(f'{indent}{dirname}/')
    for file in sorted(files):
        if file == 'README.md':
            continue
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        total_bytes += size
        ext = os.path.splitext(file)[1]
        print(f'{indent}  {file:40s} {size:>8,} bytes  [{ext}]')

print(f"\n  Total Bronze: {total_bytes:,} bytes")
print(f"  Fuentes: 4 (ERP csv, CRM json, Server logs, Encuestas xlsx)")
print(f"  Formatos: 4 diferentes")
print(f"  Esquema: ninguno — cada fuente tiene su propia estructura")

### Schema-on-Read: el problema

Cada fuente tiene columnas distintas, formatos distintos y convenciones distintas. Eso es Schema-on-Read: los datos estan ahi pero no se pueden unir sin trabajo.

In [ ]:
# ============================================================
# EL PROBLEMA: columnas incompatibles entre fuentes
# ============================================================

df_erp = pd.read_csv(f'{LAKE_ROOT}/bronze/ventas_erp/2024-06-01.csv')
df_crm = pd.read_json(f'{LAKE_ROOT}/bronze/crm_api/deals_2024-06.json')

print("Columnas del ERP:")
print(f"  {df_erp.columns.tolist()}")

print("\nColumnas del CRM:")
print(f"  {df_crm.columns.tolist()}")

print("\nProblemas para unir:")
print("  - ERP dice 'producto', CRM dice 'product_interest'")
print("  - ERP dice 'region', CRM dice 'city'")
print("  - ERP tiene precio por unidad, CRM tiene valor total del deal")
print("  - Las fechas tienen formato distinto")
print("\nEsto se resuelve en Silver.")

---
## 3. Silver: limpiar y unificar

In [ ]:
# ============================================================
# SILVER: PROCESAR VENTAS DEL ERP
# ============================================================

# Leer todos los CSVs del ERP y unificar en un solo DataFrame

os.makedirs(f'{LAKE_ROOT}/silver/ventas', exist_ok=True)

archivos_erp = sorted([
    f for f in os.listdir(f'{LAKE_ROOT}/bronze/ventas_erp')
    if f.endswith('.csv')
])

dfs = []
for archivo in archivos_erp:
    ruta = f'{LAKE_ROOT}/bronze/ventas_erp/{archivo}'
    df = pd.read_csv(ruta)
    df['archivo_origen'] = archivo  # Metadata: de donde vino
    dfs.append(df)

df_ventas = pd.concat(dfs, ignore_index=True)

# Limpiar
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_ventas['producto'] = df_ventas['producto'].str.strip().str.lower()
df_ventas['region'] = df_ventas['region'].str.strip().str.lower()
df_ventas['ingreso'] = df_ventas['unidades'] * df_ventas['precio']
df_ventas = df_ventas.drop_duplicates()

# Guardar como Parquet (formato eficiente)
ruta_silver = f'{LAKE_ROOT}/silver/ventas/ventas_consolidadas.parquet'
df_ventas.to_parquet(ruta_silver, index=False)

print(f"Silver ventas: {len(df_ventas)} filas")
print(f"Guardado: {ruta_silver}")
print(f"\nColumnas: {df_ventas.columns.tolist()}")
print(f"Tipos:\n{df_ventas.dtypes}")

In [ ]:
# ============================================================
# SILVER: PROCESAR LOGS DEL SERVIDOR
# ============================================================

# Parsear texto semi-estructurado a un DataFrame limpio

import re

os.makedirs(f'{LAKE_ROOT}/silver/logs', exist_ok=True)

with open(f'{LAKE_ROOT}/bronze/server_logs/app_2024-06-01.log') as f:
    lineas = f.readlines()

registros = []
for linea in lineas:
    linea = linea.strip()
    if not linea:
        continue
    
    # Parsear la estructura: timestamp nivel user=X action=Y ...
    match = re.match(
        r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+(\w+)\s+user=(\w+)\s+action=(\w+)(.*)',
        linea
    )
    if match:
        timestamp, nivel, usuario, accion, extra = match.groups()
        registros.append({
            'timestamp': timestamp,
            'nivel': nivel,
            'usuario': usuario,
            'accion': accion,
            'detalle': extra.strip(),
        })

df_logs = pd.DataFrame(registros)
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])

ruta_logs_silver = f'{LAKE_ROOT}/silver/logs/logs_parseados.parquet'
df_logs.to_parquet(ruta_logs_silver, index=False)

print(f"Silver logs: {len(df_logs)} eventos parseados")
print(f"Guardado: {ruta_logs_silver}")
df_logs

In [ ]:
# ============================================================
# SILVER: PROCESAR ENCUESTAS
# ============================================================

os.makedirs(f'{LAKE_ROOT}/silver/encuestas', exist_ok=True)

df_enc = pd.read_excel(f'{LAKE_ROOT}/bronze/encuestas/nps_junio_2024.xlsx')

# Estandarizar nombres de columnas
df_enc = df_enc.rename(columns={
    'Fecha Respuesta': 'fecha',
    'Cliente': 'cliente',
    'Producto Evaluado': 'producto',
    'NPS (0-10)': 'nps',
    'Comentario': 'comentario',
})

df_enc['fecha'] = pd.to_datetime(df_enc['fecha'], dayfirst=True)
df_enc['producto'] = df_enc['producto'].str.strip().str.lower()
df_enc['cliente'] = df_enc['cliente'].str.strip().str.lower()

# Clasificar NPS: Detractor (0-6), Pasivo (7-8), Promotor (9-10)
df_enc['categoria_nps'] = pd.cut(
    df_enc['nps'],
    bins=[-1, 6, 8, 10],
    labels=['detractor', 'pasivo', 'promotor']
)

ruta_enc_silver = f'{LAKE_ROOT}/silver/encuestas/nps_limpio.parquet'
df_enc.to_parquet(ruta_enc_silver, index=False)

print(f"Silver encuestas: {len(df_enc)} respuestas")
df_enc

---
## 4. Gold: agregar para consumo

In [ ]:
# ============================================================
# GOLD: RESUMEN DE VENTAS POR REGION Y PRODUCTO
# ============================================================

os.makedirs(f'{LAKE_ROOT}/gold/dashboard_ventas', exist_ok=True)

# Leer desde Silver (ya limpio)
df = pd.read_parquet(f'{LAKE_ROOT}/silver/ventas/ventas_consolidadas.parquet')

# Agregar por region
resumen_region = (df.groupby('region')
    .agg(
        transacciones=('ingreso', 'count'),
        ingreso_total=('ingreso', 'sum'),
        ticket_promedio=('ingreso', 'mean'),
        unidades_total=('unidades', 'sum'),
    )
    .round(2)
    .sort_values('ingreso_total', ascending=False)
    .reset_index())

resumen_region.to_parquet(f'{LAKE_ROOT}/gold/dashboard_ventas/resumen_regional.parquet', index=False)

# Agregar por producto
resumen_producto = (df.groupby('producto')
    .agg(
        transacciones=('ingreso', 'count'),
        ingreso_total=('ingreso', 'sum'),
        precio_promedio=('precio', 'mean'),
    )
    .round(2)
    .sort_values('ingreso_total', ascending=False)
    .reset_index())

resumen_producto.to_parquet(f'{LAKE_ROOT}/gold/dashboard_ventas/resumen_producto.parquet', index=False)

print("=== Gold: resumen regional ===")
print(resumen_region.to_string(index=False))
print("\n=== Gold: resumen por producto ===")
print(resumen_producto.to_string(index=False))

In [ ]:
# ============================================================
# GOLD: KPIs DIARIOS
# ============================================================

kpis_diarios = (df.groupby('fecha')
    .agg(
        ingreso=('ingreso', 'sum'),
        transacciones=('ingreso', 'count'),
        ticket_promedio=('ingreso', 'mean'),
        unidades=('unidades', 'sum'),
    )
    .round(2)
    .reset_index())

kpis_diarios.to_parquet(f'{LAKE_ROOT}/gold/dashboard_ventas/kpis_diarios.parquet', index=False)

print("=== Gold: KPIs diarios ===")
print(kpis_diarios.to_string(index=False))

In [ ]:
# ============================================================
# VISTA COMPLETA DEL LAKE
# ============================================================

print("Estructura completa del Data Lake:\n")
for root, dirs, files in os.walk(LAKE_ROOT):
    level = root.replace(LAKE_ROOT, '').count(os.sep)
    indent = '  ' * level
    dirname = os.path.basename(root)
    # Colorear por zona
    zona_tag = ''
    if 'bronze' in root:
        zona_tag = ' [CRUDO]'
    elif 'silver' in root:
        zona_tag = ' [LIMPIO]'
    elif 'gold' in root:
        zona_tag = ' [CONSUMO]'
    print(f'{indent}{dirname}/{zona_tag}')
    subindent = '  ' * (level + 1)
    for file in sorted(files):
        if file == 'README.md':
            continue
        size = os.path.getsize(os.path.join(root, file))
        print(f'{subindent}{file} ({size:,} bytes)')

---
## 5. Particionamiento

Cuando un dataset tiene millones de filas, particionarlo por una columna clave (fecha, region) hace que las consultas lean solo la carpeta relevante.

In [ ]:
# ============================================================
# GENERAR DATASET GRANDE PARA DEMOSTRAR PARTICIONAMIENTO
# ============================================================

np.random.seed(42)
n = 500_000

df_grande = pd.DataFrame({
    'fecha': pd.date_range('2023-01-01', periods=n, freq='min'),
    'producto': np.random.choice(['dashboard', 'reporte', 'api', 'app_web'], n),
    'region': np.random.choice(['bogota', 'medellin', 'cali', 'manizales'], n, p=[0.4, 0.25, 0.2, 0.15]),
    'unidades': np.random.randint(1, 50, n),
    'precio': np.round(np.random.uniform(100, 800, n), 2),
})
df_grande['ingreso'] = df_grande['unidades'] * df_grande['precio']
df_grande['anio'] = df_grande['fecha'].dt.year
df_grande['mes'] = df_grande['fecha'].dt.month

print(f"Dataset grande: {n:,} filas, {len(df_grande.columns)} columnas")
print(f"Memoria: {df_grande.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# ============================================================
# GUARDAR SIN PARTICIONES vs CON PARTICIONES
# ============================================================

# Sin particiones: un solo archivo
ruta_sin = 'test_sin_particion.parquet'
df_grande.to_parquet(ruta_sin, index=False)
size_sin = os.path.getsize(ruta_sin)
print(f"Sin particiones: {size_sin / 1e6:.1f} MB (1 archivo)")

# Con particiones: una carpeta por anio y mes
ruta_con = 'test_con_particion'
if os.path.exists(ruta_con):
    shutil.rmtree(ruta_con)

df_grande.to_parquet(ruta_con, index=False, partition_cols=['anio', 'mes'])

# Contar archivos y tamano
total_size = 0
total_files = 0
for root, dirs, files in os.walk(ruta_con):
    for f in files:
        if f.endswith('.parquet'):
            total_size += os.path.getsize(os.path.join(root, f))
            total_files += 1

print(f"Con particiones: {total_size / 1e6:.1f} MB ({total_files} archivos)")

# Mostrar estructura de carpetas
print("\nEstructura:")
for root, dirs, files in os.walk(ruta_con):
    level = root.replace(ruta_con, '').count(os.sep)
    if level <= 2:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 2:
            for f in files[:1]:  # Solo mostrar 1 archivo por carpeta
                print(f'{indent}  {f}')

In [ ]:
# ============================================================
# BENCHMARK: leer con y sin particiones
# ============================================================

# Consulta: solo datos de junio 2023

# Sin particiones: lee TODO el archivo y filtra
t0 = time.time()
df_sin = pd.read_parquet(ruta_sin)
df_sin = df_sin[(df_sin['anio'] == 2023) & (df_sin['mes'] == 6)]
t_sin = time.time() - t0

# Con particiones: lee SOLO la carpeta anio=2023/mes=6
t0 = time.time()
df_con = pd.read_parquet(
    ruta_con,
    filters=[('anio', '=', 2023), ('mes', '=', 6)]
)
t_con = time.time() - t0

print(f"Consulta: datos de junio 2023")
print(f"  Resultado: {len(df_con):,} filas\n")
print(f"  Sin particiones: {t_sin:.3f} seg (leyo {n:,} filas, filtro despues)")
print(f"  Con particiones: {t_con:.3f} seg (leyo solo {len(df_con):,} filas)")
print(f"  Speedup: {t_sin / t_con:.1f}x mas rapido")

---
## 6. Formatos: CSV vs Parquet

In [ ]:
# ============================================================
# BENCHMARK COMPLETO: tamano, escritura, lectura
# ============================================================

resultados = []

# CSV
t0 = time.time()
df_grande.to_csv('bench.csv', index=False)
tw_csv = time.time() - t0
size_csv = os.path.getsize('bench.csv')

t0 = time.time()
_ = pd.read_csv('bench.csv')
tr_csv = time.time() - t0

resultados.append({
    'formato': 'CSV', 'tamano_mb': round(size_csv / 1e6, 1),
    'escribir_seg': round(tw_csv, 2), 'leer_seg': round(tr_csv, 2),
    'compresion': 'No', 'tipos_preservados': 'No',
})

# Parquet (snappy - default)
t0 = time.time()
df_grande.to_parquet('bench.parquet')
tw_pq = time.time() - t0
size_pq = os.path.getsize('bench.parquet')

t0 = time.time()
_ = pd.read_parquet('bench.parquet')
tr_pq = time.time() - t0

resultados.append({
    'formato': 'Parquet (snappy)', 'tamano_mb': round(size_pq / 1e6, 1),
    'escribir_seg': round(tw_pq, 2), 'leer_seg': round(tr_pq, 2),
    'compresion': 'Si', 'tipos_preservados': 'Si',
})

# Parquet (gzip - mas compresion)
t0 = time.time()
df_grande.to_parquet('bench_gz.parquet', compression='gzip')
tw_gz = time.time() - t0
size_gz = os.path.getsize('bench_gz.parquet')

t0 = time.time()
_ = pd.read_parquet('bench_gz.parquet')
tr_gz = time.time() - t0

resultados.append({
    'formato': 'Parquet (gzip)', 'tamano_mb': round(size_gz / 1e6, 1),
    'escribir_seg': round(tw_gz, 2), 'leer_seg': round(tr_gz, 2),
    'compresion': 'Si (mayor)', 'tipos_preservados': 'Si',
})

df_bench = pd.DataFrame(resultados)
print(f"Benchmark con {n:,} filas:\n")
df_bench

In [ ]:
# ============================================================
# TIPOS PRESERVADOS: la diferencia practica
# ============================================================

# CSV pierde los tipos
df_csv = pd.read_csv('bench.csv')
print("Despues de leer CSV:")
print(f"  fecha:    {df_csv['fecha'].dtype}  ← string, hay que convertir")
print(f"  unidades: {df_csv['unidades'].dtype}")
print(f"  precio:   {df_csv['precio'].dtype}")

# Parquet preserva los tipos
df_pq = pd.read_parquet('bench.parquet')
print(f"\nDespues de leer Parquet:")
print(f"  fecha:    {df_pq['fecha'].dtype}  ← datetime, listo para usar")
print(f"  unidades: {df_pq['unidades'].dtype}")
print(f"  precio:   {df_pq['precio'].dtype}")

---
## 7. Almacenamiento columnar: lectura selectiva

In [ ]:
# ============================================================
# LEER SOLO LAS COLUMNAS QUE NECESITAS
# ============================================================

# CSV lee TODAS las columnas siempre
t0 = time.time()
df_csv_full = pd.read_csv('bench.csv')
t_csv = time.time() - t0

# Parquet lee solo las columnas pedidas
t0 = time.time()
df_pq_2cols = pd.read_parquet('bench.parquet', columns=['producto', 'ingreso'])
t_pq_2 = time.time() - t0

t0 = time.time()
df_pq_todas = pd.read_parquet('bench.parquet')
t_pq_all = time.time() - t0

print(f"Leer 500K filas:")
print(f"  CSV (8 columnas, lee todo):      {t_csv:.3f} seg")
print(f"  Parquet (8 columnas, lee todo):   {t_pq_all:.3f} seg")
print(f"  Parquet (2 columnas, selectivo):  {t_pq_2:.3f} seg")
print(f"\nCSV lee 8 columnas aunque solo necesites 2.")
print(f"Parquet lee solo las 2 que pediste.")
print(f"Con un dataset de 50 columnas la diferencia es enorme.")

In [ ]:
# ============================================================
# POR QUE FUNCIONA: filas vs columnas
# ============================================================

print("""
CSV guarda por FILAS (row-based):
─────────────────────────────────
  Ana, Bogota, dashboard, 15, 450.00
  Carlos, Medellin, api, 8, 620.00
  Maria, Cali, reporte, 22, 280.00

  Para leer solo 'ingreso', tiene que recorrer TODA la fila
  y saltar los campos que no necesita. Con 50 columnas, lee
  50 campos por cada fila para quedarse con 1.

Parquet guarda por COLUMNAS (columnar):
───────────────────────────────────────
  [Ana, Carlos, Maria]           ← bloque 'nombre'
  [Bogota, Medellin, Cali]       ← bloque 'region'
  [dashboard, api, reporte]      ← bloque 'producto'
  [15, 8, 22]                    ← bloque 'unidades'
  [450.00, 620.00, 280.00]       ← bloque 'precio'

  Para leer solo 'precio', lee SOLO ese bloque.
  Los otros 4 bloques ni se tocan.
  Con 50 columnas y necesitas 2: lee el 4% del archivo.
""")

---
## 8. Catalogo de datos

In [ ]:
# ============================================================
# CATALOGO: registrar que hay en el Lake
# ============================================================

# Un catalogo previene el Data Swamp:
# cada dataset tiene nombre, descripcion, ubicacion, propietario

def registrar_en_catalogo(catalogo, nombre, zona, ruta, descripcion,
                          propietario, formato, filas=None, columnas=None):
    """Agrega una entrada al catalogo de datos."""
    catalogo.append({
        'nombre': nombre,
        'zona': zona,
        'ruta': ruta,
        'descripcion': descripcion,
        'propietario': propietario,
        'formato': formato,
        'filas': filas,
        'columnas': columnas,
        'registrado_el': datetime.now().isoformat(),
    })

catalogo = []

registrar_en_catalogo(catalogo,
    nombre='ventas_erp_diarias',
    zona='bronze',
    ruta='bronze/ventas_erp/*.csv',
    descripcion='Ventas diarias del sistema ERP. Un archivo CSV por dia.',
    propietario='equipo-ingenieria',
    formato='CSV',
)

registrar_en_catalogo(catalogo,
    nombre='ventas_consolidadas',
    zona='silver',
    ruta='silver/ventas/ventas_consolidadas.parquet',
    descripcion='Ventas del ERP unificadas, limpias y tipadas.',
    propietario='equipo-datos',
    formato='Parquet',
    filas=len(df_ventas),
    columnas=df_ventas.columns.tolist(),
)

registrar_en_catalogo(catalogo,
    nombre='logs_parseados',
    zona='silver',
    ruta='silver/logs/logs_parseados.parquet',
    descripcion='Logs del servidor parseados a columnas estructuradas.',
    propietario='equipo-infra',
    formato='Parquet',
    filas=len(df_logs),
    columnas=df_logs.columns.tolist(),
)

registrar_en_catalogo(catalogo,
    nombre='resumen_regional',
    zona='gold',
    ruta='gold/dashboard_ventas/resumen_regional.parquet',
    descripcion='Ingreso y transacciones agregados por region. Para el dashboard de ventas.',
    propietario='equipo-datos',
    formato='Parquet',
    filas=len(resumen_region),
    columnas=resumen_region.columns.tolist(),
)

# Guardar el catalogo
with open(f'{LAKE_ROOT}/catalogo.json', 'w') as f:
    json.dump(catalogo, f, indent=2)

print("Catalogo de datos:\n")
df_catalogo = pd.DataFrame(catalogo)[['nombre', 'zona', 'formato', 'propietario', 'filas']]
df_catalogo

In [ ]:
# ============================================================
# BUSCAR EN EL CATALOGO
# ============================================================

def buscar_dataset(catalogo, termino):
    """Busca datasets en el catalogo por nombre o descripcion."""
    resultados = []
    for entry in catalogo:
        if (termino.lower() in entry['nombre'].lower() or
            termino.lower() in entry['descripcion'].lower()):
            resultados.append(entry)
    return resultados

# "Donde estan los datos de ventas?"
print("Busqueda: 'ventas'\n")
for r in buscar_dataset(catalogo, 'ventas'):
    print(f"  [{r['zona']:6s}] {r['nombre']}")
    print(f"           {r['descripcion']}")
    print(f"           Ruta: {r['ruta']}")
    print(f"           Propietario: {r['propietario']}")
    print()

In [ ]:
# Limpiar archivos de prueba
for path in [LAKE_ROOT, 'test_sin_particion.parquet', 'test_con_particion',
             'bench.csv', 'bench.parquet', 'bench_gz.parquet']:
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.isfile(path):
        os.remove(path)

print("Archivos de prueba eliminados.")

---
## Resumen

| Concepto | Lo que importa |
|---|---|
| **Bronze** | Datos crudos, formato original. Nunca se modifican. Fuente de verdad |
| **Silver** | Limpios, tipados, deduplicados. Un DataFrame por fuente, todo en Parquet |
| **Gold** | Agregados para un caso de uso: dashboard, KPI, modelo. Lo que consume el usuario final |
| **Schema-on-Read** | Cada fuente llega con su propia estructura. La unificacion pasa en Silver, no al ingestar |
| **Particionamiento** | Dividir por fecha/region para que las consultas lean solo la carpeta relevante |
| **Parquet > CSV** | Mas pequeno, mas rapido, preserva tipos, lectura selectiva de columnas |
| **Catalogo** | Registrar que hay en el Lake, donde esta, quien lo mantiene. Previene el Data Swamp |

### Siguiente paso
En el notebook de **Lakehouse** veremos como agregar transacciones ACID y control de esquema sobre estos mismos archivos Parquet usando Delta Lake.